# NoveltyWekan ISIC Pipeline
End-to-end semi-supervised medical image segmentation for ISIC 2018 skin lesion data.
Based on the BUSI pipeline — adapted for ISIC dataset with small unlabeled pool (≤100 WL).


## Section 1: Mount Drive & Path Setup

In [ ]:
# =================================================================
# SECTION 1: Mount Drive & Setup
# =================================================================
from google.colab import drive
drive.mount('/content/drive')

import os

medsam_checkpoint = '/content/drive/MyDrive/MEDSAM/medsam_vit_b.pth'
assert os.path.exists(medsam_checkpoint), (
    'ERROR: medsam_vit_b.pth not found at /content/drive/MyDrive/MEDSAM/'
)
print('medsam_vit_b.pth found.')
print('Drive mounted and paths verified.')

## Section 2: Clone Repositories & Install Dependencies

In [ ]:
%cd /content

!rm -rf Auto-Generate-WLs
!git clone https://github.com/stanfordmlgroup/Auto-Generate-WLs

!rm -rf MedSAM
!git clone https://github.com/bowang-lab/MedSAM

!pip install -q segment-anything albumentations==1.3.1 scikit-image scikit-learn tqdm

# Fix albumentations API in cloned repos (newer albumentations changed its imports)
!find /content/Auto-Generate-WLs/pytorch-nested-unet -name '*.py' \
    -exec sed -i 's/from albumentations.augmentations import transforms/import albumentations as transforms/g' {} +
!find /content/Auto-Generate-WLs/pytorch-nested-unet -name '*.py' \
    -exec sed -i 's/from albumentations.core.composition import Compose, OneOf/from albumentations import Compose, OneOf/g' {} +
!find /content/Auto-Generate-WLs/pytorch-nested-unet -name '*.py' \
    -exec sed -i 's/from albumentations.core.composition import Compose/from albumentations import Compose/g' {} +

print('Setup complete.')

## Section 3: Verify Environment

In [ ]:
import albumentations as A
import numpy as np, cv2, torch

print('albumentations :', A.__version__)
print('numpy          :', np.__version__)
print('torch          :', torch.__version__)
print('CUDA available :', torch.cuda.is_available())

dummy = np.zeros((256, 256, 3), dtype=np.uint8)
out = A.Compose([A.Resize(128,128), A.HorizontalFlip(p=0.5)])(image=dummy)
print('Albumentations smoke test passed. Output shape:', out['image'].shape)

## Section 4: Download & Split ISIC 2018 Dataset
Splits into:
- **25 GS** (gold-standard with masks)
- **≤200 Unlabeled** (hard cap to keep Colab alive)
- **Test set** (~20% held-out with masks)

> Setup Kaggle: `!mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/`

In [ ]:
import kagglehub, os

path = kagglehub.dataset_download(
    'tschandl/isic2018-challenge-task1-data-segmentation'
)
print('Downloaded to:', path)

In [ ]:
import os, random, shutil

random.seed(42)

IMG_DIR  = os.path.join(path, 'ISIC2018_Task1-2_Training_Input')
MASK_DIR = os.path.join(path, 'ISIC2018_Task1_Training_GroundTruth')
assert os.path.exists(IMG_DIR),  f'Missing: {IMG_DIR}'
assert os.path.exists(MASK_DIR), f'Missing: {MASK_DIR}'

BASE = '/content/Auto-Generate-WLs/pytorch-nested-unet/inputs'

# Clean old splits
for d in ['isic-gs-25', 'isic-unlabeled', 'isic-test']:
    shutil.rmtree(os.path.join(BASE, d), ignore_errors=True)

for sub in [('isic-gs-25','images'), ('isic-gs-25','masks','0'),
            ('isic-unlabeled','images'),
            ('isic-test','images'), ('isic-test','masks','0')]:
    os.makedirs(os.path.join(BASE, *sub), exist_ok=True)

all_imgs = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith('.jpg')])
random.shuffle(all_imgs)

# Keep small to avoid RAM crash: 25 GS + 200 unlabeled + ~20% test
MAX_UNLABELED = 200
n_test        = max(50, int(0.20 * len(all_imgs)))
test_imgs     = all_imgs[:n_test]
train_imgs    = all_imgs[n_test:]
gs_imgs       = train_imgs[:25]
unlabeled_imgs = train_imgs[25:25 + MAX_UNLABELED]

print(f'Total: {len(all_imgs)}')
print(f'GS: {len(gs_imgs)}  Unlabeled (capped): {len(unlabeled_imgs)}  Test: {len(test_imgs)}')

def copy_split(filelist, img_dst, mask_dst=None):
    for img_name in filelist:
        shutil.copy(os.path.join(IMG_DIR, img_name), os.path.join(img_dst, img_name))
        if mask_dst:
            src_m = img_name.replace('.jpg', '_segmentation.png')
            dst_m = img_name.replace('.jpg', '.png')
            src_path = os.path.join(MASK_DIR, src_m)
            if os.path.exists(src_path):
                shutil.copy(src_path, os.path.join(mask_dst, dst_m))

copy_split(gs_imgs,        os.path.join(BASE,'isic-gs-25','images'),     os.path.join(BASE,'isic-gs-25','masks','0'))
copy_split(unlabeled_imgs, os.path.join(BASE,'isic-unlabeled','images'))
copy_split(test_imgs,      os.path.join(BASE,'isic-test','images'),       os.path.join(BASE,'isic-test','masks','0'))

# Leakage check
test_set = set(test_imgs); gs_set = set(gs_imgs); ul_set = set(unlabeled_imgs)
print(f'\n=== LEAKAGE CHECK ===')
print(f'Test ∩ GS:         {len(test_set & gs_set)}  (must be 0)')
print(f'Test ∩ Unlabeled:  {len(test_set & ul_set)}  (must be 0)')
print(f'GS ∩ Unlabeled:    {len(gs_set & ul_set)}  (must be 0)')
if not (test_set & gs_set) and not (test_set & ul_set):
    print('✅ ZERO LEAKAGE CONFIRMED.')
else:
    print('🚨 LEAKAGE DETECTED!')

## Section 5: Contrastive Pretraining of Encoder (Idea 8)
Self-supervised warmup on ≤200 unlabeled images. Fast since dataset is small.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset as TDS, DataLoader
from PIL import Image
import os, numpy as np

class ContrastiveTransform:
    def __init__(self, size=128):
        self.t = T.Compose([
            T.RandomResizedCrop(size, scale=(0.5,1.0)),
            T.RandomHorizontalFlip(),
            T.ColorJitter(0.4,0.4,0.4,0.1),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
        ])
    def __call__(self, x):
        return self.t(x), self.t(x)

class UnlabeledDS(TDS):
    def __init__(self, img_dir, transform=None):
        self.paths = sorted([os.path.join(img_dir,f) for f in os.listdir(img_dir)
                             if f.lower().endswith(('.jpg','.jpeg','.png'))])
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img) if self.transform else T.ToTensor()(img)

class SimpleEncoder(nn.Module):
    def __init__(self, out=128):
        super().__init__()
        self.bb = nn.Sequential(
            nn.Conv2d(3,32,3,stride=2,padding=1), nn.ReLU(),
            nn.Conv2d(32,64,3,stride=2,padding=1), nn.ReLU(),
            nn.Conv2d(64,128,3,stride=2,padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4,4)))
        self.proj = nn.Sequential(nn.Flatten(), nn.Linear(2048,256), nn.ReLU(), nn.Linear(256,out))
    def forward(self,x): return self.proj(self.bb(x))

def nt_xent(z1, z2, temp=0.5):
    z1 = nn.functional.normalize(z1,dim=1); z2 = nn.functional.normalize(z2,dim=1)
    N = z1.size(0); z = torch.cat([z1,z2])
    sim = torch.mm(z, z.T) / temp
    sim.masked_fill_(torch.eye(2*N, dtype=torch.bool, device=z.device), float('-inf'))
    labels = torch.cat([torch.arange(N,2*N), torch.arange(N)]).to(z.device)
    return nn.CrossEntropyLoss()(sim, labels)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ul_dir = '/content/Auto-Generate-WLs/pytorch-nested-unet/inputs/isic-unlabeled/images'
ds = UnlabeledDS(ul_dir, ContrastiveTransform(128))
loader = DataLoader(ds, batch_size=32, shuffle=True, num_workers=2, drop_last=True)
enc = SimpleEncoder().to(device)
opt = optim.Adam(enc.parameters(), lr=3e-4)

enc.train()
for ep in range(5):
    tot = 0
    for v1, v2 in loader:
        v1, v2 = v1.to(device), v2.to(device)
        loss = nt_xent(enc(v1), enc(v2))
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()
    print(f'Epoch {ep+1}/5  loss={tot/len(loader):.4f}')
print('Contrastive pretraining done.')

## Section 6: Seed Synchronization & In-Place train.py Patch
Adds --seed argument to train.py parser.

In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet

with open('train.py', 'r') as f:
    tr = f.read()
tr = tr.replace('\r\n', '\n')

if '--seed' not in tr:
    tr = tr.replace(
        "parser.add_argument('--num_workers', default=4, type=int)",
        "parser.add_argument('--num_workers', default=4, type=int)\n"
        "    parser.add_argument('--seed', default=42, type=int)"
    )
    seed_code = """
    import random, numpy as np, torch
    seed = config['seed']
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
"""
    tr = tr.replace('config = vars(parse_args())', 'config = vars(parse_args())' + seed_code)
    with open('train.py', 'w') as f: f.write(tr)
    print('train.py patched with --seed support.')
else:
    print('--seed already in train.py.')

%cd /content

## Section 7: Train Baseline NestedUNet on 25 GS Images

In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet
!python train.py \
    --dataset isic-gs-25 \
    --name isic-gs-25_NestedUNet_256_woDS \
    --arch NestedUNet \
    --img_ext .jpg --mask_ext .png \
    --batch_size 4 --input_w 256 --input_h 256 \
    --seed 42 --optimizer Adam --lr 3e-4 --epochs 200
%cd /content

## Section 8: Evaluate Baseline Model on isic-test

In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet
!python eval.py \
    --name isic-gs-25_NestedUNet_256_woDS \
    --dataset isic-test \
    --img_ext .jpg --mask_ext .png
%cd /content

## Section 9: Pre-Coarse Complexity Gate (Idea 1)
Trains a Random Forest on GS images to filter easy unlabeled images first.

In [ ]:
complexity_gate_src = """
import cv2
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def compute_complexity_score(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f'Image not found: {image_path}')
    img = cv2.resize(img, (256, 256))
    normalized = img / 255.0
    hist, _ = np.histogram(normalized.ravel(), bins=32, density=True)
    hist = hist + 1e-10
    entropy = -np.sum(hist * np.log(hist))
    edges = cv2.Canny(img, 50, 150)
    edge_density = np.sum(edges > 0) / (256 * 256)
    local_variance = np.std(normalized)
    border = np.concatenate([normalized[0,:], normalized[-1,:], normalized[:,0], normalized[:,-1]])
    bg_homogeneity = 1.0 / (np.std(border) + 1e-5)
    return {
        'local_variance': local_variance,
        'edge_density': edge_density,
        'entropy': entropy,
        'bg_homogeneity': bg_homogeneity,
    }

def train_complexity_classifier(gs_image_paths, gs_mask_paths, medsam_predictions):
    X, y = [], []
    for img_p, gt_p, pred_mask in zip(gs_image_paths, gs_mask_paths, medsam_predictions):
        feats = compute_complexity_score(img_p)
        X.append(list(feats.values()))
        gt = (cv2.imread(gt_p, cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
        pr = (pred_mask > 127).astype(np.uint8)
        inter = np.logical_and(gt, pr).sum()
        dice = 2. * inter / (gt.sum() + pr.sum() + 1e-6)
        y.append(1 if dice > 0.5 else 0)
    sc = StandardScaler()
    X_scaled = sc.fit_transform(X)
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    clf.fit(X_scaled, y)
    return clf, sc

def gate_images(image_paths, clf, scaler, threshold=0.5):
    approved, rejected = [], []
    for path in image_paths:
        try:
            feats = list(compute_complexity_score(path).values())
            feats_s = scaler.transform([feats])
            if len(clf.classes_) >= 2:
                prob = clf.predict_proba(feats_s)[0][1]
            else:
                prob = 1.0 if clf.classes_[0] == 1 else 0.0
            (approved if prob >= threshold else rejected).append(path)
        except Exception:
            rejected.append(path)
    if len(approved) < 40:
        def prob_fn(p):
            try:
                f = scaler.transform([list(compute_complexity_score(p).values())])
                return clf.predict_proba(f)[0][1] if len(clf.classes_)>=2 else float(clf.classes_[0]==1)
            except: return 0.
        ranked = sorted(image_paths, key=prob_fn, reverse=True)
        approved, rejected = ranked[:40], ranked[40:]
    return approved, rejected
"""

import os
os.makedirs('Auto-Generate-WLs/generate-weak-labels', exist_ok=True)
with open('Auto-Generate-WLs/generate-weak-labels/complexity_gate.py', 'w') as f:
    f.write(complexity_gate_src)
print('complexity_gate.py written.')

## Section 10: Augmentation Consistency Check TTA (Idea 3)

In [ ]:
import torch
import torchvision.transforms.functional as TF
import numpy as np
import cv2
from scipy import ndimage

def augmentation_consistency_check(image, model, device, n_augments=6, iou_threshold=0.20):
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    model.eval()
    boxes = []

    # SPEED: resize to 256x256 BEFORE GPU ops — critical for large ISIC images
    small = cv2.resize(image, (256, 256))
    t_base = torch.tensor(small.transpose(2,0,1), dtype=torch.float32).unsqueeze(0).to(device) / 255.

    for angle in np.linspace(-30, 30, n_augments):
        t = TF.rotate(t_base, float(angle))
        with torch.no_grad():
            p = model(t)
            if isinstance(p, list): p = p[-1]
            p = torch.sigmoid(p).squeeze().cpu().numpy()

        p_bin = (p > 0.5).astype(np.uint8)
        p_rot = (ndimage.rotate(p_bin, -float(angle), reshape=False, order=0) > 0.5).astype(np.uint8)
        cs, _ = cv2.findContours(p_rot, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cs:
            x, y, w, h = cv2.boundingRect(max(cs, key=cv2.contourArea))
            boxes.append([x, y, x+w, y+h])

    if len(boxes) < n_augments // 4:
        return False, 0.

    ious = []
    for i in range(len(boxes)):
        for j in range(i+1, len(boxes)):
            b1, b2 = boxes[i], boxes[j]
            ix = max(0, min(b1[2],b2[2]) - max(b1[0],b2[0]))
            iy = max(0, min(b1[3],b2[3]) - max(b1[1],b2[1]))
            inter = ix * iy
            union = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
            ious.append(inter / (union + 1e-5))

    m = float(np.mean(ious)) if ious else 0.
    return m >= iou_threshold, m

## Section 11: Setup MedSAM Model & Refinement (Idea 5)

In [ ]:
import torch
import cv2
import numpy as np
from segment_anything import sam_model_registry

device = 'cuda' if torch.cuda.is_available() else 'cpu'
medsam_ckpt = '/content/drive/MyDrive/MEDSAM/medsam_vit_b.pth'
assert os.path.exists(medsam_ckpt), 'medsam_vit_b.pth not found!'

medsam_model = sam_model_registry['vit_b'](checkpoint=medsam_ckpt).to(device)
medsam_model.eval()
print(f'MedSAM loaded on {device}')

def run_medsam_refinement(image_bgr, coarse_mask):
    H, W = image_bgr.shape[:2]
    binary = (coarse_mask > 127).astype(np.uint8)
    ys, xs = np.where(binary > 0)
    if len(xs) == 0:
        return None
    x1, y1, x2, y2 = xs.min(), ys.min(), xs.max(), ys.max()
    if (x2 - x1) < 5 or (y2 - y1) < 5:
        return None

    # MedSAM requires EXACTLY 1024x1024 input — scale then pad
    scale = 1024 / max(H, W)
    H1024 = int(round(H * scale))
    W1024 = int(round(W * scale))
    img_rs = cv2.resize(image_bgr, (W1024, H1024))

    # Pad to exactly 1024x1024
    padded = np.zeros((1024, 1024, 3), dtype=np.uint8)
    padded[:H1024, :W1024] = img_rs

    img_1024 = padded.astype(np.float32) / 255.
    img_1024 = (img_1024 - img_1024.min()) / np.clip(img_1024.max() - img_1024.min(), 1e-8, None)
    img_1024 = np.transpose(img_1024, (2, 0, 1))
    img_tensor = torch.tensor(img_1024).unsqueeze(0).to(device)

    # Scale bounding box to 1024 padded space
    box_1024 = np.clip(
        np.array([x1*scale, y1*scale, x2*scale, y2*scale], dtype=np.float32),
        0, 1023
    )

    with torch.no_grad():
        image_embedding = medsam_model.image_encoder(img_tensor)
        box_torch = torch.tensor(box_1024, dtype=torch.float32).to(device).unsqueeze(0).unsqueeze(0)
        sparse_emb, dense_emb = medsam_model.prompt_encoder(
            points=None, boxes=box_torch, masks=None)
        low_res_logits, _ = medsam_model.mask_decoder(
            image_embeddings=image_embedding,
            image_pe=medsam_model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_emb,
            dense_prompt_embeddings=dense_emb,
            multimask_output=False)

    probs = torch.sigmoid(low_res_logits).cpu().numpy().squeeze()
    # Crop padding back out, then resize to original
    probs = probs[:H1024, :W1024]
    probs = cv2.resize(probs, (W, H))
    probs = np.clip(probs, 0, 1)
    return (probs > 0.5).astype(np.uint8) * 255

## Section 12: V2 Morphological Gate Filters (Idea 2)

In [ ]:
def morphological_gate_v2(coarse_mask, image_shape=(256, 256)):
    mask = (coarse_mask > 127).astype(np.uint8)
    H, W = mask.shape[:2]

    fill_ratio = (mask > 0).mean()
    if fill_ratio > 0.97 or fill_ratio < 0.002:
        return False, f'extreme_fill ({fill_ratio:.3f})', None

    num_labels, labels = cv2.connectedComponents(mask)
    if num_labels <= 1:
        return False, 'no_object', None

    component_sizes = [np.sum(labels == i) for i in range(1, num_labels)]
    significant_sizes = [s for s in component_sizes if s >= 50]
    if len(significant_sizes) > 200:
        return False, f'too_fragmented ({len(significant_sizes)})', None

    largest_label = 1 + np.argmax(component_sizes)
    clean_mask = (labels == largest_label).astype(np.uint8)

    sorted_sig = sorted(significant_sizes, reverse=True)
    if len(sorted_sig) >= 2 and sorted_sig[1] / (sorted_sig[0] + 1e-5) > 0.75:
        return False, f'ambiguous_components ({sorted_sig[1]/(sorted_sig[0]+1e-5):.2f})', None

    contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return False, 'no_contour', None

    main_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(main_contour)
    aspect_ratio = max(w,h) / (min(w,h) + 1e-5)
    if aspect_ratio > 15.0:
        return False, f'bad_aspect ({aspect_ratio:.1f})', None

    area = cv2.contourArea(main_contour)
    if area < 0.0005 * H * W:
        return False, f'too_small ({area:.0f})', None

    hull = cv2.convexHull(main_contour)
    convexity = area / (cv2.contourArea(hull) + 1e-5)
    if convexity < 0.03:
        return False, f'low_convexity ({convexity:.2f})', None

    ys_c, xs_c = np.where(clean_mask > 0)
    if len(xs_c) > 0:
        stripe_ratio = len(np.unique(xs_c)) / (len(np.unique(ys_c)) + 1e-5)
        if stripe_ratio > 7.0:
            return False, f'stripe_artifact ({stripe_ratio:.1f})', None

    return True, 'passed', clean_mask * 255

## Section 13: Radiomics Quality Predictor (Idea 7)

In [ ]:
from skimage.measure import regionprops
from skimage.feature import graycomatrix, graycoprops
from sklearn.ensemble import GradientBoostingRegressor

def extract_skimage_features(image, mask):
    mask_bin = (mask > 127).astype(np.uint8)
    props = regionprops(mask_bin)
    if len(props) == 0:
        return {'area': 0., 'eccentricity': 0., 'perimeter': 0., 'mean_intensity': 0., 'contrast': 0.}
    prop = max(props, key=lambda x: x.area)
    img_clipped = (image * 255).astype(np.uint8) if image.max() <= 1.0 else image.astype(np.uint8)
    glcm = graycomatrix(img_clipped, distances=[2], angles=[0], levels=256, symmetric=True, normed=True)
    contrast = float(graycoprops(glcm, 'contrast')[0, 0])
    return {
        'area': float(prop.area),
        'eccentricity': float(prop.eccentricity),
        'perimeter': float(prop.perimeter),
        'mean_intensity': float(np.mean(image[mask_bin > 0]) if mask_bin.sum() > 0 else 0.),
        'contrast': contrast,
    }

def train_radiomics_quality_model(gs_image_paths, gs_mask_paths, medsam_predictions):
    X, y = [], []
    for img_p, gt_p, pred_mask in zip(gs_image_paths, gs_mask_paths, medsam_predictions):
        img = cv2.imread(img_p, cv2.IMREAD_GRAYSCALE)
        gt  = (cv2.imread(gt_p, cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
        feats = extract_skimage_features(img, pred_mask)
        X.append(list(feats.values()))
        inter = np.logical_and(gt, (pred_mask > 127)).sum()
        dice = 2. * inter / (gt.sum() + (pred_mask > 127).sum() + 1e-5)
        y.append(dice)
    model = GradientBoostingRegressor(n_estimators=30, random_state=42)
    model.fit(X, y)
    return model

## Section 14: Curriculum Sorting & Multi-Gate Weak Label Generation (Idea 5)
1. Complexity Gate filters unlabeled pool
2. TTA Consistency Check (Idea 3)
3. MedSAM Refinement (Idea 5)
4. Morphological Gate V2 (Idea 2)
5. Radiomics Quality Check (Idea 7)
Cap: 100 weak labels maximum.

In [ ]:
import sys, os, shutil, torch, cv2
import numpy as np
from collections import Counter
from tqdm import tqdm

sys.path.append('Auto-Generate-WLs/generate-weak-labels')
import complexity_gate, importlib
importlib.reload(complexity_gate)
from complexity_gate import compute_complexity_score, train_complexity_classifier, gate_images

%cd /content/Auto-Generate-WLs/pytorch-nested-unet
import archs

device = 'cuda' if torch.cuda.is_available() else 'cpu'
coarse_model_path = 'models/isic-gs-25_NestedUNet_256_woDS/model.pth'
coarse_net = archs.NestedUNet(num_classes=1, input_channels=3, deep_supervision=False).to(device)
coarse_net.load_state_dict(torch.load(coarse_model_path, map_location=device))
coarse_net.eval()
print('Coarse model loaded.')

GS_IMG_DIR   = 'inputs/isic-gs-25/images'
GS_MASK_DIR  = 'inputs/isic-gs-25/masks/0'
UL_IMG_DIR   = 'inputs/isic-unlabeled/images'

gs_img_paths  = sorted([os.path.join(GS_IMG_DIR, f) for f in os.listdir(GS_IMG_DIR) if f.lower().endswith(('.jpg','.png'))])
gs_mask_paths = [os.path.join(GS_MASK_DIR, os.path.splitext(os.path.basename(f))[0]+'.png') for f in gs_img_paths]

unlabeled_files = sorted([f for f in os.listdir(UL_IMG_DIR) if f.lower().endswith('.jpg')])
print(f'GS: {len(gs_img_paths)}  Unlabeled: {len(unlabeled_files)}')

def coarse_predict_small(img_bgr):
    """Predict at 256x256 — fast and memory-safe for large ISIC images."""
    small = cv2.resize(img_bgr, (256, 256))
    t = torch.tensor(small.transpose(2,0,1)).unsqueeze(0).float().to(device) / 255.
    with torch.no_grad():
        p = torch.sigmoid(coarse_net(t)).squeeze().cpu().numpy()
    H, W = img_bgr.shape[:2]
    coarse_mask = (cv2.resize(p, (W,H)) > 0.5).astype(np.uint8) * 255
    return coarse_mask, p

# Generate coarse predictions on GS to train gate models
print('Generating coarse predictions for GS images...')
gs_preds = []
for ip in tqdm(gs_img_paths):
    img = cv2.imread(ip)
    if img is None:
        gs_preds.append(np.zeros((256,256), np.uint8)); continue
    cm, _ = coarse_predict_small(img)
    ref = run_medsam_refinement(img, cm)
    gs_preds.append(ref if ref is not None else cm)

clf_comp, scaler_comp = train_complexity_classifier(gs_img_paths, gs_mask_paths, gs_preds)
quality_regressor = train_radiomics_quality_model(gs_img_paths, gs_mask_paths, gs_preds)
print('Gate models trained.')

# Coarse-scan unlabeled (store only the tiny 256x256 score, not full-res mask)
print('Coarse-scanning unlabeled images...')
unlabeled_paths, coarse_preds = [], []
for fn in tqdm(unlabeled_files):
    img_path = os.path.join(UL_IMG_DIR, fn)
    img = cv2.imread(img_path)
    if img is None: continue
    small = cv2.resize(img, (256, 256))
    t = torch.tensor(small.transpose(2,0,1)).unsqueeze(0).float().to(device) / 255.
    with torch.no_grad():
        p = torch.sigmoid(coarse_net(t)).squeeze().cpu().numpy()
    unlabeled_paths.append(img_path)
    coarse_preds.append(p)
    del img, small, t, p  # free immediately

# Curriculum sort: easy images first
confidence_scores = [pred.max() - abs((pred>0.5).mean() - 0.15) for pred in coarse_preds]
sorted_pairs = sorted(zip(confidence_scores, unlabeled_paths), key=lambda x: x[0], reverse=True)
sorted_paths = [p for _, p in sorted_pairs]
del coarse_preds  # free memory

# Complexity gate
approved_paths, rejected_paths = gate_images(sorted_paths, clf_comp, scaler_comp, threshold=0.15)
print(f'Complexity Gate: Approved {len(approved_paths)}, Rejected {len(rejected_paths)}')

# Output directories
save_wl_mask_dir = 'inputs/isic-25-wl-v2/masks/0'
save_wl_img_dir  = 'inputs/isic-25-wl-v2/images'
os.makedirs(save_wl_mask_dir, exist_ok=True)
os.makedirs(save_wl_img_dir,  exist_ok=True)

MAX_WEAK_LABELS = 100   # Keep small — same as BUSI pipeline
saved_wl_names = []
skipped_counts = {}

print(f'Generating weak labels (cap={MAX_WEAK_LABELS})...')
for img_path in tqdm(approved_paths):
    if len(saved_wl_names) >= MAX_WEAK_LABELS:
        print(f'✅ Reached cap of {MAX_WEAK_LABELS} weak labels. Stopping early.')
        break

    fname = os.path.basename(img_path)
    image = cv2.imread(img_path)
    if image is None: continue

    coarse_mask, _ = coarse_predict_small(image)

    # Gate 2: TTA Consistency Check
    is_consistent, iou_val = augmentation_consistency_check(image, coarse_net, device, n_augments=6, iou_threshold=0.20)
    if not is_consistent:
        skipped_counts[fname] = f'inconsistent_TTA ({iou_val:.2f})'; continue

    # Gate 3: MedSAM Refinement
    refined_mask = run_medsam_refinement(image, coarse_mask)
    if refined_mask is None:
        skipped_counts[fname] = 'medsam_failed'; continue

    # Gate 4: Morphological Gate V2
    passed_morph, morph_reason, cleaned_mask = morphological_gate_v2(refined_mask)
    if not passed_morph:
        skipped_counts[fname] = f'morph_{morph_reason}'; continue

    # Gate 5: Radiomics Quality Check
    feats = extract_skimage_features(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY), refined_mask)
    pred_quality = quality_regressor.predict([list(feats.values())])[0]
    if pred_quality < 0.03:
        skipped_counts[fname] = f'low_quality ({pred_quality:.2f})'; continue

    # Save weak label
    mask_name = os.path.splitext(fname)[0] + '.png'
    cv2.imwrite(os.path.join(save_wl_mask_dir, mask_name), cleaned_mask)
    shutil.copy(img_path, os.path.join(save_wl_img_dir, fname))
    saved_wl_names.append(fname)

print(f'\nLoop finished. Saved {len(saved_wl_names)} weak labels.')
print('Skipped reasons:')
print(Counter(skipped_counts.values()))

In [ ]:
# Diagnostic: Count saved weak labels
import os
wl_mask_dir = 'Auto-Generate-WLs/pytorch-nested-unet/inputs/isic-25-wl-v2/masks/0'
if os.path.exists(wl_mask_dir):
    n = len(os.listdir(wl_mask_dir))
    print(f'Weak labels saved: {n}')
else:
    print('ERROR: WL directory missing — check Section 14 ran without errors.')

## Section 15: Distance-Weighted Loss Retraining (Idea 6)
Patches losses.py and train.py to down-weight noisy weak labels during training.

In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet
import os

# Reset to clean git state, then re-apply albumentations fix
os.system('git checkout -- train.py dataset.py losses.py')
os.system("find . -name '*.py' -exec sed -i 's/from albumentations.augmentations import transforms/import albumentations as transforms/g' {} +")
os.system("find . -name '*.py' -exec sed -i 's/from albumentations.core.composition import Compose, OneOf/from albumentations import Compose, OneOf/g' {} +")
os.system("find . -name '*.py' -exec sed -i 's/from albumentations.core.composition import Compose/from albumentations import Compose/g' {} +")
print('Git checkout done. Albumentations fix re-applied.')

# 1. Patch dataset.py: add is_weak flag
with open('dataset.py') as f: ds = f.read()
if 'is_weak' not in ds:
    ds = ds.replace(
        "return img, mask, {'img_id': img_id}",
        "is_weak = 1. if 'wl' in img_id or 'weak' in img_id else 0.\n"
        "        return img, mask, {'img_id': img_id, 'is_weak': is_weak}"
    )
    with open('dataset.py', 'w') as f: f.write(ds)
    print('dataset.py patched.')
else:
    print('dataset.py already patched.')

# 2. Patch losses.py: add DistanceWeightedLoss and DiceLoss
losses_code = """

import torch
import torch.nn as nn
import numpy as np
from scipy.ndimage import distance_transform_edt

class DistanceWeightedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce_loss_raw = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, output, target, is_weak_batch=None):
        batch_weights = []
        for i in range(target.shape[0]):
            if is_weak_batch is not None and is_weak_batch[i] > 0.5:
                gt_np = target[i, 0].detach().cpu().numpy() > 0.5
                if gt_np.sum() > 0:
                    d_in  = distance_transform_edt(gt_np)
                    d_out = distance_transform_edt(~gt_np)
                    d = d_in + d_out
                    d = d / d.max() if d.max() > 0 else d
                    weight = 0.4 + 0.6 * d
                else:
                    weight = np.ones_like(gt_np, dtype=np.float32)
                batch_weights.append(torch.tensor(weight, dtype=torch.float32).unsqueeze(0))
            else:
                batch_weights.append(torch.ones(1, *target.shape[2:], dtype=torch.float32))
        pixel_weights = torch.stack(batch_weights).to(output.device)
        bce_map = self.bce_loss_raw(output, target)
        weighted_bce = bce_map * pixel_weights
        if is_weak_batch is not None:
            image_weights = (1.0 - 0.5 * is_weak_batch).view(-1, 1, 1, 1)
            weighted_bce = weighted_bce * image_weights
        return weighted_bce.mean()

class DiceLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, input, target, is_weak_batch=None):
        smooth = 1e-5
        input = torch.sigmoid(input)
        num = target.size(0)
        input = input.view(num, -1)
        target = target.view(num, -1)
        intersection = (input * target)
        dice = (2. * intersection.sum(1) + smooth) / (input.sum(1) + target.sum(1) + smooth)
        loss_per_image = 1 - dice
        if is_weak_batch is not None:
            weights = 1.0 - 0.5 * is_weak_batch
            return (loss_per_image * weights).mean()
        return loss_per_image.mean()
"""
with open('losses.py', 'a') as f: f.write(losses_code)
print('losses.py patched.')

# 3. Patch train.py: seed + val_dataset + DWL
with open('train.py', 'r') as f: tr = f.read()
tr = tr.replace('\r\n', '\n')

if '--seed' not in tr:
    tr = tr.replace(
        "parser.add_argument('--num_workers', default=4, type=int)",
        "parser.add_argument('--num_workers', default=4, type=int)\n"
        "    parser.add_argument('--seed', default=42, type=int)\n"
        "    parser.add_argument('--val_dataset', default=None, type=str)"
    )
    seed_code = """
    import random
    import numpy as np
    import torch
    seed = config['seed']
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
"""
    tr = tr.replace('config = vars(parse_args())', 'config = vars(parse_args())' + seed_code)

if 'val_dataset' not in tr:
    tr = tr.replace(
        '    img_ids = sorted(img_ids)\n    train_img_ids, val_img_ids = train_test_split(img_ids, test_size=0.2, random_state=41)',
        """    img_ids = sorted(img_ids)
    if config.get('val_dataset'):
        train_img_ids = img_ids
        val_img_ids = sorted([
            os.path.splitext(fn)[0]
            for fn in os.listdir(os.path.join('inputs', config['val_dataset'], 'images'))
            if fn.endswith('.png') or fn.endswith('.jpg')
        ])
        print(f"[val_dataset] Train: {len(train_img_ids)} | Val (clean GS): {len(val_img_ids)}")
    else:
        train_img_ids, val_img_ids = train_test_split(img_ids, test_size=0.2, random_state=41)"""
    )

if 'DistanceWeightedLoss' not in tr:
    tr = tr.replace(
        'import losses',
        'import losses\nfrom losses import DistanceWeightedLoss\ndw_loss_fn = DistanceWeightedLoss()'
    ).replace(
        'for input, target, _ in train_loader:',
        'for input, target, meta in train_loader:'
    ).replace(
        '        else:\n            output = model(input)\n            loss = criterion(output, target)',
        '        else:\n            output = model(input)\n'
        '            is_weak_batch = meta.get(\'is_weak\', torch.zeros(target.shape[0])).to(target.device)\n'
        '            bce_loss = dw_loss_fn(output, target, is_weak_batch)\n'
        '            dice_loss = losses.DiceLoss()(output, target, is_weak_batch)\n'
        '            loss = dice_loss + 0.5 * bce_loss'
    )

with open('train.py', 'w') as f: f.write(tr)
print('train.py patched with: seed, val_dataset, DistanceWeightedLoss.')
%cd /content

## Section 16: Build Augmented Dataset & Retrain Model
Combines 20 GS training images with 100 weak labels. 5 clean GS images are kept as validation only.

In [ ]:
import os, shutil

gs_img_dir   = 'Auto-Generate-WLs/pytorch-nested-unet/inputs/isic-gs-25/images'
gs_mask_dir  = 'Auto-Generate-WLs/pytorch-nested-unet/inputs/isic-gs-25/masks/0'
wl_img_dir   = 'Auto-Generate-WLs/pytorch-nested-unet/inputs/isic-25-wl-v2/images'
wl_mask_dir  = 'Auto-Generate-WLs/pytorch-nested-unet/inputs/isic-25-wl-v2/masks/0'

aug_train_img  = 'Auto-Generate-WLs/pytorch-nested-unet/inputs/isic-aug-train/images'
aug_train_mask = 'Auto-Generate-WLs/pytorch-nested-unet/inputs/isic-aug-train/masks/0'
aug_val_img    = 'Auto-Generate-WLs/pytorch-nested-unet/inputs/isic-aug-val/images'
aug_val_mask   = 'Auto-Generate-WLs/pytorch-nested-unet/inputs/isic-aug-val/masks/0'

for d in [aug_train_img, aug_train_mask, aug_val_img, aug_val_mask]:
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

# Split 25 GS: first 5 → val (clean), rest 20 → train
gs_files = sorted(os.listdir(gs_img_dir))
gs_val_files   = gs_files[:5]
gs_train_files = gs_files[5:]

# Copy GS val (5 clean images — no weak labels!)
for f in gs_val_files:
    stem = os.path.splitext(f)[0]
    shutil.copy(os.path.join(gs_img_dir,  f),          os.path.join(aug_val_img,  f))
    shutil.copy(os.path.join(gs_mask_dir, stem+'.png'), os.path.join(aug_val_mask, stem+'.png'))

# Copy GS train (20 images)
for f in gs_train_files:
    stem = os.path.splitext(f)[0]
    shutil.copy(os.path.join(gs_img_dir,  f),          os.path.join(aug_train_img,  f))
    shutil.copy(os.path.join(gs_mask_dir, stem+'.png'), os.path.join(aug_train_mask, stem+'.png'))

# Copy weak labels (img_ext .jpg → training, mask_ext .png → training)
wl_files = sorted(os.listdir(wl_mask_dir)) if os.path.exists(wl_mask_dir) else []
for mf in wl_files:
    stem   = os.path.splitext(mf)[0]
    img_f  = stem + '.jpg'
    src_i  = os.path.join(wl_img_dir,  img_f)
    src_m  = os.path.join(wl_mask_dir, mf)
    if not os.path.exists(src_i): continue
    shutil.copy(src_i, os.path.join(aug_train_img,  f'wl_{img_f}'))
    shutil.copy(src_m, os.path.join(aug_train_mask, f'wl_{mf}'))

print(f'Training set : {len(os.listdir(aug_train_img))} images (20 GS + {len(wl_files)} WL)')
print(f'Validation set: {len(os.listdir(aug_val_img))} images (5 clean GS — zero weak labels)')

%cd /content/Auto-Generate-WLs/pytorch-nested-unet
!python train.py \
    --dataset isic-aug-train \
    --val_dataset isic-aug-val \
    --name isic-aug-train_NestedUNet_256_woDS \
    --arch NestedUNet \
    --img_ext .jpg --mask_ext .png \
    --batch_size 4 --input_w 256 --input_h 256 \
    --seed 42 --optimizer Adam --lr 3e-4 --epochs 200
%cd /content

## Section 17: Final Evaluation & Comparisons

In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet

print('\n' + '='*55)
print('EVALUATION: UNSEEN TEST SET (isic-test)')
print('='*55)

print('\n--- 1. BASELINE (25 GS only) ---')
!python eval.py \
    --name isic-gs-25_NestedUNet_256_woDS \
    --dataset isic-test \
    --img_ext .jpg --mask_ext .png

print('\n--- 2. IMPROVED (20 GS train + WL, clean val, DWL) ---')
!python eval.py \
    --name isic-aug-train_NestedUNet_256_woDS \
    --dataset isic-test \
    --img_ext .jpg --mask_ext .png

%cd /content

## Section 18: Test-Time Augmentation (TTA) Evaluation

In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet
import os

eval_tta_code = """
import argparse, os
from glob import glob
import numpy as np, torch, torch.backends.cudnn as cudnn, yaml
import albumentations as transforms
from albumentations import Resize, Normalize, Compose
from tqdm import tqdm
import archs
from dataset import Dataset

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--name', required=True)
    p.add_argument('--dataset', required=True)
    p.add_argument('--img_ext', default='.jpg')
    p.add_argument('--mask_ext', default='.png')
    p.add_argument('--tta_mode', default='horizontal', choices=['none','horizontal','all'])
    return vars(p.parse_args())

def main():
    args = parse_args()
    with open(f"models/{args['name']}/config.yml") as f:
        config = yaml.load(f, Loader=yaml.FullLoader)
    cudnn.benchmark = True
    model = archs.__dict__[config['arch']](
        num_classes=config['num_classes'],
        input_channels=config['input_channels'],
        deep_supervision=config['deep_supervision']).cuda()
    model.load_state_dict(torch.load(f"models/{args['name']}/model.pth"))
    model.eval()
    img_ids = [os.path.splitext(os.path.basename(p))[0]
               for p in glob(f"inputs/{args['dataset']}/images/*{args['img_ext']}")]
    val_transform = Compose([Resize(config['input_h'], config['input_w']), Normalize()])
    val_dataset = Dataset(
        img_ids=img_ids,
        img_dir=f"inputs/{args['dataset']}/images",
        mask_dir=f"inputs/{args['dataset']}/masks",
        img_ext=args['img_ext'], mask_ext=args['mask_ext'],
        num_classes=config['num_classes'], transform=val_transform)
    ious, dices = [], []
    smooth = 1e-5
    with torch.no_grad():
        for i in tqdm(range(len(val_dataset))):
            img, gt, meta = val_dataset[i]
            it  = torch.from_numpy(img).unsqueeze(0).cuda()
            gt_t = torch.from_numpy(gt).unsqueeze(0).cuda()
            if config['deep_supervision']:
                p0 = torch.sigmoid(model(it)[-1])
            else:
                p0 = torch.sigmoid(model(it))
            if args['tta_mode'] == 'none':
                pm = p0
            elif args['tta_mode'] == 'horizontal':
                ih = torch.flip(it, dims=[3])
                ph = torch.sigmoid(model(ih)[-1] if config['deep_supervision'] else model(ih))
                ph = torch.flip(ph, dims=[3])
                pm = (p0 + ph) / 2.
            else:
                ih = torch.flip(it, dims=[3])
                ph = torch.sigmoid(model(ih)[-1] if config['deep_supervision'] else model(ih))
                ph = torch.flip(ph, dims=[3])
                iv = torch.flip(it, dims=[2])
                pv = torch.sigmoid(model(iv)[-1] if config['deep_supervision'] else model(iv))
                pv = torch.flip(pv, dims=[2])
                ihv = torch.flip(it, dims=[2,3])
                phv = torch.sigmoid(model(ihv)[-1] if config['deep_supervision'] else model(ihv))
                phv = torch.flip(phv, dims=[2,3])
                pm = (p0 + ph + pv + phv) / 4.
            pb = pm > 0.5; gb = gt_t > 0.5
            ious.append(((pb & gb).sum().float() + smooth) / ((pb | gb).sum().float() + smooth))
            dices.append((2.*(pm*gt_t).sum() + smooth) / (pm.sum() + gt_t.sum() + smooth))
    print(f"TTA Mode: {args['tta_mode']}")
    print(f"TTA IoU:  {np.mean([v.item() for v in ious]):.4f}")
    print(f"TTA Dice: {np.mean([v.item() for v in dices]):.4f}")

if __name__ == '__main__': main()
"""

with open('eval_tta.py', 'w') as f:
    f.write(eval_tta_code)
print('eval_tta.py written.')

for model_name, label in [
    ('isic-gs-25_NestedUNet_256_woDS',     'BASELINE'),
    ('isic-aug-train_NestedUNet_256_woDS', 'IMPROVED'),
]:
    for mode in ['none', 'horizontal', 'all']:
        print(f'\n=== {label} | TTA={mode} ===')
        os.system(
            f'python eval_tta.py '
            f'--name {model_name} '
            f'--dataset isic-test '
            f'--img_ext .jpg --mask_ext .png '
            f'--tta_mode {mode}'
        )

%cd /content

## Section 19: Save Models to Drive (Optional)

In [ ]:
import os, shutil
os.makedirs('/content/drive/MyDrive/models', exist_ok=True)

!cp -r /content/Auto-Generate-WLs/pytorch-nested-unet/models/isic-gs-25_NestedUNet_256_woDS \
        /content/drive/MyDrive/models/
!cp -r /content/Auto-Generate-WLs/pytorch-nested-unet/models/isic-aug-train_NestedUNet_256_woDS \
        /content/drive/MyDrive/models/

print('Models saved to Google Drive.')